# Week 5 Session: Model Tracking and Deployment

> **Configuration Note:** This notebook uses the project data folders and the saved model checkpoint from previous sessions. If a path does not resolve, check that the project files and data folder are mounted in the expected course directory.

In week 3 session 4, we already saw the key modeling problem: our XGBoost misinformation detector performs reasonably on the original FakeNewsNet split, but struggles on the Bloomberg/GPT-4o shifted dataset. We will not treat that failure as a new discovery today. Instead, we will use it as the starting point for a production question:

**When a model fails, gets retrained, and is replaced, how do we know exactly what happened?**

So far, many results have lived inside notebook variables. That is fine while exploring, but risky for production ML. A result depends on the exact data version, feature schema, parameters, code, and model artifact used at the time.

> **Important:** In a real ML project, you usually introduce tracking from the beginning. We delayed it in this course project to keep the first modeling notebooks focused on data preparation, baselines, robustness, and interpretation. Today we are adding the tool after the fact so you can clearly see the problem it solves. In future projects, start tracking before the experiments become hard to reconstruct.

We will:
- Recreate the **week 4 XGBoost production baseline**.
- Reuse the **Bloomberg/GPT-4o failure case** from week 3 session 4 as motivation.
- Retrain candidate models after adding part of the shifted data.
- Use **Weights & Biases (W&B)** to track data versions, feature schemas, metrics, artifacts, and trained models.
- Compare runs and recover the exact setup behind a result.
- Load a tracked model and call it through a deployment-style prediction interface.

## Learning Objectives & Flow

By the end of this session you should be able to:

1. **Explain why model tracking matters** - connect the week 3 session 4 distribution-shift failure to the need for reproducible model history.
2. **Use a known failure case as a tracking problem** - revisit the Bloomberg/GPT-4o result without re-teaching the full shift analysis.
3. **Recognize why feature schemas must be stable** - understand why the preprocessing implementation remains important even when the concept is already familiar.
4. **Track experiments with W&B** - log parameters, metrics, artifacts, model files, dataset fingerprints, and feature lists.
5. **Compare model versions** - decide which run should become the next candidate model using logged evidence.
6. **Recover model history** - answer which data, features, and parameters produced a specific result.
7. **Load a tracked model for prediction** - use the selected tracked model artifact through a deployment-style interface.

The main idea is that today is not about discovering distribution shift from scratch. Today is about making the response to that shift traceable, and about seeing why future projects should start with tracking earlier.

## Setup

First, we install and import the libraries used in this session.

The important new tool is **Weights & Biases (W&B)**. We will use it for:
- **Experiment tracking:** storing parameters, metrics, and artifacts for every model run.
- **Hosted run comparison:** using the W&B web dashboard without local port forwarding.
- **Artifact logging:** saving feature schemas, dataset fingerprints, confusion matrices, and model files.
- **Model loading:** downloading a selected model artifact later and using it for prediction.

W&B does not replace a production model-serving system by itself. Instead, it gives us the tracked model artifact and lineage we would use when deploying with a separate serving tool.

We also use the same modeling stack as previous sessions: pandas, scikit-learn, XGBoost, NLTK, textstat, matplotlib, and seaborn.

In [ ]:
# Package setup for reproducibility.
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "wandb", "xgboost", "nltk", "textstat", "shap",
    "scikit-learn", "matplotlib", "seaborn",
    "fastapi", "uvicorn",
])


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import sys
import tempfile
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
from IPython.display import display


### Project Configuration

Like the earlier misinformation notebooks, this notebook uses the project `config.py` file for paths.

The configuration gives us:

- `BASE_PATH`: the root of the misinformation project
- `DATA_PATH`: the folder containing processed and raw data
- `SRC_PATH`: the folder containing the `misinformation_detection` package
- `SAVED_CHECKPOINTS_PATH`: the folder used for saved model checkpoints

This keeps the notebook consistent with the rest of the project and avoids hard-coding multiple possible folder layouts.

In [ ]:
from google.colab import userdata

config_path = userdata.get("CONFIG_PATH")
if config_path and config_path not in sys.path:
    sys.path.append(config_path)

from config import BASE_PATH, DATA_PATH, SRC_PATH, SAVED_CHECKPOINTS_PATH

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

PROJECT_ROOT = BASE_PATH
CHECKPOINT_PATH = SAVED_CHECKPOINTS_PATH

# Some previous week notebooks stored checkpoints directly under BASE_PATH/saved_checkpoints.
# Prefer config.py, but allow that existing project convention if present.
if not CHECKPOINT_PATH.exists() and (BASE_PATH / "saved_checkpoints").exists():
    CHECKPOINT_PATH = BASE_PATH / "saved_checkpoints"

FNN_PATH = DATA_PATH / "processed" / "fnn_lemmatized.csv"
BLOOMBERG_PATH = DATA_PATH / "raw" / "bloomberg_fake_news_dataset.csv"

print("BASE_PATH:", BASE_PATH)
print("DATA_PATH:", DATA_PATH)
print("SRC_PATH:", SRC_PATH)
print("SAVED_CHECKPOINTS_PATH:", SAVED_CHECKPOINTS_PATH)
print("FNN_PATH exists:", FNN_PATH.exists(), FNN_PATH)
print("BLOOMBERG_PATH exists:", BLOOMBERG_PATH.exists(), BLOOMBERG_PATH)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)


In [ ]:
import nltk
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.utils.validation import check_is_fitted
from xgboost import XGBClassifier
import joblib

for resource in [
    "punkt",
    "punkt_tab",
    "wordnet",
    "stopwords",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
]:
    nltk.download(resource, quiet=True)

from misinformation_detection.data.tracking_features import (
    BASE_STRUCTURED_COLUMNS,
    DRIFT_COLUMNS_TO_IGNORE,
    INVERSE_LABEL_MAP,
    LABEL_MAP,
    MAX_TFIDF_FEATURES,
    RANDOM_STATE,
    build_external_feature_matrix,
    canonicalize_feature_frame,
    clean_and_lemmatize_articles,
    dataset_fingerprint,
    load_bloomberg_shift,
    matrix_from_feature_frame,
)

XGB_PARAMS = {
    "n_estimators": 100,
    "max_depth": 5,
    "learning_rate": 0.1,
    "eval_metric": "logloss",
    "random_state": RANDOM_STATE,
}


## Part 1: Feature-Building Helpers for Reproducibility

You have already worked through the feature pipeline in earlier sessions: text cleaning, lemmatization, linguistic features, source-domain features, TF-IDF features, and column alignment.

Today, that pipeline is not the main topic. We still need it because tracking only works if the model input is reproducible. A tracked run is much less useful if we cannot answer which exact columns entered the model.

To keep the notebook focused on tracking and deployment, the week 5 feature helpers live in `src/misinformation_detection/data/tracking_features.py`. They do five practical things:

1. Reuse the familiar preprocessing steps from earlier weeks.
2. Create a stable training feature schema.
3. Align external evaluation data to that exact schema.
4. Avoid accidental schema drift from temporary notebook columns such as `predicted_label`, `group_weight`, or `source_domain.1`.
5. Create dataset fingerprints and feature-schema metadata that we can log to W&B.

This is a common production pattern: preprocessing code belongs in reusable project code, while the notebook shows the workflow and the evidence we want to track.


In [ ]:
print("Week 5 feature helpers imported from src.")
print("Base structured feature count:", len(BASE_STRUCTURED_COLUMNS))
print("Columns explicitly blocked from accidental schema drift:")
for column in sorted(DRIFT_COLUMNS_TO_IGNORE):
    print(" -", column)


## Part 2: Recreate the Current Production Baseline

### Why start here?

We are continuing the story from previous sessions. The XGBoost model is our current production baseline, and the Bloomberg/GPT-4o result is already known to be problematic.

Before we introduce W&B tracking, we intentionally recreate the old workflow once more. This gives us the situation that motivates tracking: a model exists, a result exists, but the history behind that result is easy to lose if it only lives in notebook state.

The production model uses two types of signals:

- **TF-IDF text features:** word-level patterns from lemmatized article text.
- **Structured features:** linguistic and source metadata such as readability, source domain, punctuation, and clickbait indicators.

In this section we:
1. Load the processed FakeNewsNet data.
2. Build the canonical train/test feature matrices.
3. Load the existing XGBoost checkpoint if it matches the feature schema.
4. Otherwise train the same XGBoost configuration to keep the workflow reproducible.
5. Evaluate the model on the original FakeNewsNet holdout set.

The important question is not just "what score did we get?" The important question is: **can we later recover exactly how this score was produced?**

In [ ]:
fnn_df = pd.read_csv(FNN_PATH)
fnn_df = fnn_df[fnn_df["label"].isin(["real", "fake"])].copy()

print("FakeNewsNet shape:", fnn_df.shape)
print("Label balance:")
display(fnn_df["label"].value_counts(normalize=True).rename("share").to_frame().join(fnn_df["label"].value_counts().rename("count")))

fnn_train_df, fnn_test_df = train_test_split(
    fnn_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=fnn_df["label"],
)

fnn_train_features, SOURCE_DOMAINS, STRUCTURED_COLUMNS = canonicalize_feature_frame(fnn_train_df)
fnn_test_features, _, _ = canonicalize_feature_frame(fnn_test_df, source_domains=SOURCE_DOMAINS)

vectorizer_v1 = TfidfVectorizer(max_features=MAX_TFIDF_FEATURES, stop_words="english")
X_train_v1, y_train_v1 = matrix_from_feature_frame(
    fnn_train_features,
    vectorizer=vectorizer_v1,
    structured_columns=STRUCTURED_COLUMNS,
    fit_vectorizer=True,
)
TRAIN_FEATURE_NAMES_V1 = X_train_v1.columns.tolist()
X_test_v1, y_test_v1 = matrix_from_feature_frame(
    fnn_test_features,
    vectorizer=vectorizer_v1,
    structured_columns=STRUCTURED_COLUMNS,
    feature_names=TRAIN_FEATURE_NAMES_V1,
    fit_vectorizer=False,
)

print("Train feature matrix:", X_train_v1.shape)
print("Test feature matrix:", X_test_v1.shape)
print("Source domains:", SOURCE_DOMAINS)

In [ ]:
def evaluate_classifier(model, X, y, label="model", show_report=True):
    preds = model.predict(X)
    metrics = {
        "accuracy": accuracy_score(y, preds),
        "precision_fake": precision_score(y, preds, pos_label=1, zero_division=0),
        "recall_fake": recall_score(y, preds, pos_label=1, zero_division=0),
        "f1_fake": f1_score(y, preds, pos_label=1, zero_division=0),
    }
    print(f"{label} metrics")
    for key, value in metrics.items():
        print(f"  {key}: {value:.4f}")
    if show_report:
        print("\nClassification report")
        print(classification_report(y, preds, target_names=["real", "fake"], zero_division=0))
    return metrics, preds


def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["real", "fake"], yticklabels=["real", "fake"])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def load_or_train_production_xgb(X_train, y_train, X_probe):
    checkpoint_file = CHECKPOINT_PATH / "xgboost_model.joblib"
    if checkpoint_file.exists() and checkpoint_file.stat().st_size > 0:
        try:
            model = joblib.load(checkpoint_file)
            model.predict(X_probe.head(1))
            print(f"Loaded compatible production checkpoint: {checkpoint_file}")
            return model, "loaded_checkpoint"
        except Exception as exc:
            print(f"Checkpoint exists but is not compatible with this feature schema: {exc}")
            print("Training a reproducible XGBoost baseline instead.")

    model = XGBClassifier(**XGB_PARAMS)
    model.fit(X_train, y_train)
    return model, "trained_in_notebook"

production_xgb, production_source = load_or_train_production_xgb(X_train_v1, y_train_v1, X_test_v1)
print("Production model source:", production_source)

prod_fnn_metrics, prod_fnn_preds = evaluate_classifier(production_xgb, X_test_v1, y_test_v1, label="Production XGBoost on FakeNewsNet holdout")
plot_confusion(y_test_v1, prod_fnn_preds, "Production XGBoost - FakeNewsNet Holdout")

## Part 3: Revisit the Bloomberg/GPT-4o Failure Case

This section is a recap and continuation of week 3 session 4, not a new distribution-shift lesson.

The Bloomberg/GPT-4o file contains paired articles:

- `real_news`: original Bloomberg articles
- `fake_news`: GPT-4o-modified versions of those articles

We convert this wide format into a standard classification dataset with one article per row and labels `real` or `fake`, just as before.

### Why include it again?

Because this is the moment where tracking becomes necessary.

The model failure itself is already familiar: the production model does not generalize well to this shifted data. What we investigate today is what happens next:

- We retrain with updated data.
- We compare several model candidates.
- We create a new candidate production model.
- We need to explain later which version produced which result.

So the Bloomberg/GPT-4o dataset is our motivating example for model tracking, not the main new modeling concept.

In [ ]:
shift_df = load_bloomberg_shift(BLOOMBERG_PATH)

print("Bloomberg/GPT-4o long-format shape:", shift_df.shape)
print("Label counts:")
display(shift_df["label"].value_counts().to_frame("count"))

display(shift_df[["id", "label", "article"]].head(3))

In [ ]:
X_shift_all_v1, y_shift_all = build_external_feature_matrix(
    shift_df,
    vectorizer=vectorizer_v1,
    train_feature_names=TRAIN_FEATURE_NAMES_V1,
    source_domains=SOURCE_DOMAINS,
    structured_columns=STRUCTURED_COLUMNS,
)

print("Shift feature matrix:", X_shift_all_v1.shape)
print("Columns aligned exactly:", X_shift_all_v1.columns.tolist() == TRAIN_FEATURE_NAMES_V1)

prod_shift_metrics, prod_shift_preds = evaluate_classifier(
    production_xgb,
    X_shift_all_v1,
    y_shift_all,
    label="Production XGBoost on Bloomberg/GPT-4o shift",
)
plot_confusion(y_shift_all, prod_shift_preds, "Production XGBoost - Bloomberg/GPT-4o Shift")

### The Audit Problem

After evaluating on the shifted batch, pause and ask:

- Was this result produced by the saved checkpoint or by a retrained model?
- Which exact feature list did the model expect?
- Did we accidentally include a temporary notebook column such as `group_weight`?
- Which dataset version produced the metrics?
- Which model should be considered the current production model now?
- If we rerun this notebook next week, can we recover the same result?

This is the practical problem W&B tracking helps solve. It gives every run a record: parameters, metrics, artifacts, input schema, model file, and data version metadata.

In [ ]:
# Pick one shifted example that the production model misclassified, if available.
shift_audit_df = shift_df.copy()
shift_audit_df["y_true"] = y_shift_all.map(INVERSE_LABEL_MAP).values
shift_audit_df["y_pred"] = pd.Series(prod_shift_preds).map(INVERSE_LABEL_MAP).values
if hasattr(production_xgb, "predict_proba"):
    shift_audit_df["prob_fake"] = production_xgb.predict_proba(X_shift_all_v1)[:, 1]
else:
    shift_audit_df["prob_fake"] = np.nan

mistakes = shift_audit_df[shift_audit_df["y_true"] != shift_audit_df["y_pred"]]
audit_example = mistakes.iloc[0] if len(mistakes) else shift_audit_df.iloc[0]

print("Example to audit later")
print("id:", audit_example["id"])
print("true:", audit_example["y_true"])
print("predicted:", audit_example["y_pred"])
print("prob_fake:", audit_example["prob_fake"])
print("article excerpt:", audit_example["article"][:800])

## Part 4: Retraining Runs for Tracking

### Why retrain here?

From week 3 session 4, we already know the current model struggles on the shifted Bloomberg/GPT-4o examples. A natural next step is to update the training data and train again.

The main purpose here is **not** to re-teach model selection or decide that one model family is universally best. We already covered baselines and model tradeoffs in earlier sessions. Here, a few candidate models give us multiple W&B runs to compare, audit, and discuss.

Retraining creates a tracking problem. Once we produce a new model, we need to know:

- Which data version did it use?
- Which feature schema did it use?
- Which hyperparameters did it use?
- How did it perform on the old FakeNewsNet holdout and on the shifted data?
- Did improvement on the shifted data come with a cost on the original distribution?

Here we simulate a realistic update cycle:

1. Keep the original FakeNewsNet holdout set for in-distribution evaluation.
2. Split the Bloomberg/GPT-4o data into a small update-training portion and a shifted evaluation portion.
3. Train a few candidate models on the updated training data.
4. Compare performance on both evaluation sets.

### Important comparison caveat

This is an illustrative tracking exercise, not a perfectly controlled model-selection benchmark.

In the code below, Logistic Regression and Random Forest use `class_weight="balanced"`, while XGBoost uses the same week 4-style parameters as before. That means the comparison is not fully fair as a model-family comparison. For today's goal, that is acceptable because we mainly want several runs with different metric profiles. In a real model-selection exercise, we would tune class imbalance handling consistently across candidates.

Also watch for tradeoffs: a model can score higher on Bloomberg fake-class F1 while dropping noticeably on FakeNewsNet accuracy. W&B helps make those tradeoffs visible instead of hiding them behind a single "best run" label.

In [ ]:
shift_train_df, shift_eval_df = train_test_split(
    shift_df,
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=shift_df["label"],
)

updated_train_df = pd.concat([fnn_train_df, shift_train_df], ignore_index=True)

print("Dataset versions")
print("  fnn_v1 train rows:", len(fnn_train_df))
print("  fnn_plus_bloomberg_v2 train rows:", len(updated_train_df))
print("  held-out FNN rows:", len(fnn_test_df))
print("  held-out Bloomberg shift rows:", len(shift_eval_df))

fingerprint_fnn_v1 = dataset_fingerprint(fnn_train_df, "FakeNewsNet", "fnn_v1")
fingerprint_v2 = dataset_fingerprint(updated_train_df, "FakeNewsNet + Bloomberg/GPT-4o", "fnn_plus_bloomberg_v2")

print("fnn_v1 hash:", fingerprint_fnn_v1["sha256"][:16])
print("fnn_plus_bloomberg_v2 hash:", fingerprint_v2["sha256"][:16])

In [ ]:
def build_train_eval_matrices(train_raw_df, eval_raw_sets, source_domains=None):
    train_features, source_domains, structured_columns = canonicalize_feature_frame(train_raw_df, source_domains=source_domains)
    vectorizer = TfidfVectorizer(max_features=MAX_TFIDF_FEATURES, stop_words="english")
    X_train, y_train = matrix_from_feature_frame(
        train_features,
        vectorizer=vectorizer,
        structured_columns=structured_columns,
        fit_vectorizer=True,
    )
    feature_names = X_train.columns.tolist()

    eval_matrices = {}
    for eval_name, eval_df in eval_raw_sets.items():
        X_eval, y_eval = build_external_feature_matrix(
            eval_df,
            vectorizer=vectorizer,
            train_feature_names=feature_names,
            source_domains=source_domains,
            structured_columns=structured_columns,
        )
        eval_matrices[eval_name] = (X_eval, y_eval)

    schema = {
        "feature_names": feature_names,
        "source_domains": source_domains,
        "structured_columns": structured_columns,
        "tfidf_features": list(vectorizer.get_feature_names_out()),
    }
    return X_train, y_train, eval_matrices, vectorizer, schema


eval_sets_raw = {
    "fnn_holdout": fnn_test_df,
    "bloomberg_shift_eval": shift_eval_df,
}

X_train_v2, y_train_v2, eval_matrices_v2, vectorizer_v2, schema_v2 = build_train_eval_matrices(
    updated_train_df,
    eval_sets_raw,
    source_domains=SOURCE_DOMAINS,
)

print("Updated train matrix:", X_train_v2.shape)
for name, (X_eval, y_eval) in eval_matrices_v2.items():
    print(name, X_eval.shape, y_eval.shape)

## Part 5: W&B Tracking

### What is W&B tracking?

Weights & Biases creates a structured record of each model run. Instead of writing results manually into a table, we log them as part of the training process and inspect them in a hosted web dashboard.

In this notebook, W&B appears after several weeks of modeling because we first wanted to understand the data, the baselines, and the failure case. In a real project, once you know you will run repeated experiments, tracking should be part of the first serious training loop.

For each run, we log:

- **Configuration:** model type, hyperparameters, feature count, dataset version
- **Metrics:** accuracy, precision, recall, and F1 on each evaluation set
- **Artifacts:** confusion matrices, feature schema, dataset fingerprint
- **Model artifact:** the trained model saved in a reloadable format

### Why W&B here instead of a local MLflow UI?

A local MLflow UI requires a web server and port forwarding, which is fragile in hosted notebook environments. W&B gives us a hosted web UI for the same teaching goal: compare runs, inspect metrics, and recover artifacts later.

### Why log the dataset fingerprint?

A dataset name such as `fnn_plus_bloomberg_v2` is helpful, but it is not enough. The same name could accidentally point to a slightly changed CSV. A fingerprint hash gives us a compact way to detect whether the data content changed.

### Why log the feature schema?

The model input schema is part of the model. If two runs both say "XGBoost" but one uses 130 features and another uses 132 features, they are not the same model setup. Logging the schema makes this visible.


In [ ]:
import wandb

WANDB_PROJECT = "week5_misinformation_tracking"
WANDB_ARTIFACT_DIR = PROJECT_ROOT / "wandb_artifact_files"
WANDB_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# In a notebook, this will ask for a W&B API key if you are not already logged in.
wandb.login()
print("W&B project:", WANDB_PROJECT)


In [ ]:
def sanitize_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", str(value)).strip("-")


def write_json_artifact_file(run_id, payload, filename):
    run_dir = WANDB_ARTIFACT_DIR / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    path = run_dir / filename
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return path


def log_json_artifact(run, payload, artifact_name, artifact_type, filename):
    path = write_json_artifact_file(run.id, payload, filename)
    artifact = wandb.Artifact(name=sanitize_name(artifact_name), type=artifact_type)
    artifact.add_file(str(path))
    run.log_artifact(artifact)
    return artifact


def log_confusion_matrix(run, y_true, y_pred, artifact_name):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["real", "fake"], yticklabels=["real", "fake"], ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(artifact_name.replace("_", " "))
    fig.tight_layout()

    run.log({artifact_name: wandb.Image(fig)})

    run_dir = WANDB_ARTIFACT_DIR / run.id / "confusion_matrices"
    run_dir.mkdir(parents=True, exist_ok=True)
    path = run_dir / f"{sanitize_name(artifact_name)}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")

    artifact = wandb.Artifact(name=sanitize_name(f"confusion-matrix-{artifact_name}-{run.id}"), type="evaluation")
    artifact.add_file(str(path))
    run.log_artifact(artifact)
    plt.close(fig)


def metric_dict(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_fake": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall_fake": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1_fake": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    }


def train_evaluate_log_model(model_name, estimator, train_data, eval_sets, metadata):
    X_train, y_train = train_data
    estimator = clone(estimator)

    config = {
        "model_name": model_name,
        "dataset_version": metadata["dataset_version"],
        "feature_count": X_train.shape[1],
        "train_rows": X_train.shape[0],
        "course_week": "week_5",
    }
    params = estimator.get_params(deep=False)
    for key, value in params.items():
        if isinstance(value, (str, int, float, bool, type(None))):
            config[f"param_{key}"] = value

    run = wandb.init(
        project=WANDB_PROJECT,
        name=f"{metadata['dataset_version']}__{model_name}",
        config=config,
        reinit=True,
    )

    try:
        start = time.time()
        estimator.fit(X_train, y_train)
        train_seconds = time.time() - start
        run.log({"train_seconds": train_seconds})

        all_metrics = {}
        for eval_name, (X_eval, y_eval) in eval_sets.items():
            preds = estimator.predict(X_eval)
            metrics = metric_dict(y_eval, preds)
            prefixed_metrics = {f"{eval_name}_{metric_name}": metric_value for metric_name, metric_value in metrics.items()}
            run.log(prefixed_metrics)
            all_metrics.update(prefixed_metrics)
            log_confusion_matrix(run, y_eval, preds, f"{eval_name}_{model_name}")

        log_json_artifact(
            run,
            metadata["dataset_fingerprint"],
            artifact_name=f"dataset-fingerprint-{metadata['dataset_version']}-{run.id}",
            artifact_type="dataset_metadata",
            filename="dataset_fingerprint.json",
        )
        log_json_artifact(
            run,
            metadata["feature_schema"],
            artifact_name=f"feature-schema-{metadata['dataset_version']}-{run.id}",
            artifact_type="feature_schema",
            filename="feature_schema.json",
        )

        model_dir = WANDB_ARTIFACT_DIR / run.id / "model"
        model_dir.mkdir(parents=True, exist_ok=True)
        model_path = model_dir / "model.joblib"
        joblib.dump(estimator, model_path)

        model_artifact_name = sanitize_name(f"model-{model_name}-{run.id}")
        model_artifact = wandb.Artifact(
            name=model_artifact_name,
            type="model",
            metadata={
                "model_name": model_name,
                "dataset_version": metadata["dataset_version"],
                "feature_count": X_train.shape[1],
            },
        )
        model_artifact.add_file(str(model_path))
        run.log_artifact(model_artifact, aliases=["latest", sanitize_name(model_name), sanitize_name(metadata["dataset_version"])])

        run_path = "/".join(run.path) if isinstance(run.path, (list, tuple)) else run.path
        entity = run.entity or run_path.split("/")[0]
        project = run.project or WANDB_PROJECT
        model_artifact_ref = f"{entity}/{project}/{model_artifact_name}:latest"
        result = {
            "run_id": run.id,
            "wandb_run_path": run_path,
            "wandb_run_url": run.url,
            "model_artifact_ref": model_artifact_ref,
            "model_name": model_name,
            "dataset_version": metadata["dataset_version"],
            **all_metrics,
        }
        return result, estimator
    finally:
        run.finish()


In [ ]:
# Log the existing production-style XGBoost baseline on fnn_v1.
# This run is intentionally separate from the manual baseline above: W&B becomes the source of truth from here onward.
X_train_v1_logged, y_train_v1_logged, eval_matrices_v1, vectorizer_v1_logged, schema_v1 = build_train_eval_matrices(
    fnn_train_df,
    {
        "fnn_holdout": fnn_test_df,
        "bloomberg_shift_eval": shift_eval_df,
    },
    source_domains=SOURCE_DOMAINS,
)

run_rows = []
trained_models = {}

metadata_v1 = {
    "dataset_version": "fnn_v1",
    "dataset_fingerprint": fingerprint_fnn_v1,
    "feature_schema": schema_v1,
}

result, fitted = train_evaluate_log_model(
    "xgboost_production_baseline",
    XGBClassifier(**XGB_PARAMS),
    (X_train_v1_logged, y_train_v1_logged),
    eval_matrices_v1,
    metadata_v1,
)
run_rows.append(result)
trained_models[result["run_id"]] = fitted

pd.DataFrame(run_rows)

In [ ]:
# Compare a compact model set after adding the newly labeled Bloomberg/GPT-4o examples.
# This is mainly to create multiple W&B runs for tracking and tradeoff inspection.
metadata_v2 = {
    "dataset_version": "fnn_plus_bloomberg_v2",
    "dataset_fingerprint": fingerprint_v2,
    "feature_schema": schema_v2,
}

candidate_models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(n_estimators=150, max_depth=12, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    "xgboost": XGBClassifier(**XGB_PARAMS),
}

for model_name, estimator in candidate_models.items():
    result, fitted = train_evaluate_log_model(
        model_name,
        estimator,
        (X_train_v2, y_train_v2),
        eval_matrices_v2,
        metadata_v2,
    )
    run_rows.append(result)
    trained_models[result["run_id"]] = fitted

results_df = pd.DataFrame(run_rows)

baseline_row = results_df.query("model_name == 'xgboost_production_baseline' and dataset_version == 'fnn_v1'").iloc[0]
results_df["delta_fnn_accuracy_vs_production"] = (
    results_df["fnn_holdout_accuracy"] - baseline_row["fnn_holdout_accuracy"]
)
results_df["delta_bloomberg_f1_vs_production"] = (
    results_df["bloomberg_shift_eval_f1_fake"] - baseline_row["bloomberg_shift_eval_f1_fake"]
)

results_df = results_df.sort_values(
    ["bloomberg_shift_eval_f1_fake", "bloomberg_shift_eval_recall_fake", "fnn_holdout_f1_fake"],
    ascending=False,
)

columns_for_discussion = [
    "model_name",
    "dataset_version",
    "fnn_holdout_accuracy",
    "bloomberg_shift_eval_f1_fake",
    "bloomberg_shift_eval_recall_fake",
    "delta_fnn_accuracy_vs_production",
    "delta_bloomberg_f1_vs_production",
    "wandb_run_url",
    "run_id",
]

display(results_df[columns_for_discussion])

best_run_id = results_df.iloc[0]["run_id"]
best_model_name = results_df.iloc[0]["model_name"]
print("Selected run by shifted-data fake F1:", best_run_id)
print("Selected model by shifted-data fake F1:", best_model_name)
print("\nDiscussion note:")
print("The top shifted-data model is not automatically the best production replacement.")
print("Check delta_fnn_accuracy_vs_production to see whether shifted-data gains came with original-distribution cost.")
print("Also remember: Logistic Regression and Random Forest used class_weight='balanced'; XGBoost did not in this compact demo.")


### Exploring Runs in the W&B Web UI

W&B gives us a hosted run comparison page, so we do not need local port forwarding.

When you open the run URLs or the project dashboard, focus on these questions:

1. Which model performs best on the original FakeNewsNet holdout set?
2. Which model performs best on the Bloomberg/GPT-4o shifted set?
3. Does the best shifted-data model sacrifice too much original-distribution performance?
4. Do all runs use the dataset version and feature count we expected?
5. Which artifacts would help explain a result later?

The notebook table below mirrors the most important dashboard information, while the W&B links let you inspect the same runs in the web UI.

In [ ]:
wandb_runs_df = results_df.copy()

columns_for_wandb_view = [
    "model_name",
    "dataset_version",
    "fnn_holdout_accuracy",
    "fnn_holdout_f1_fake",
    "bloomberg_shift_eval_accuracy",
    "bloomberg_shift_eval_recall_fake",
    "bloomberg_shift_eval_f1_fake",
    "delta_fnn_accuracy_vs_production",
    "delta_bloomberg_f1_vs_production",
    "wandb_run_url",
    "model_artifact_ref",
]

display(wandb_runs_df[columns_for_wandb_view])

project_url = f"https://wandb.ai/{wandb_runs_df.iloc[0]['wandb_run_path'].split('/')[0]}/{WANDB_PROJECT}"
print("W&B project dashboard:", project_url)
print("Selected run by shifted-data fake F1:", best_run_id)
print("Selected model:", best_model_name)
print("Remember to inspect the delta columns before treating this as a production replacement.")


## Part 6: Recovering the History of a Result

Earlier, we saw why the shifted-data result is hard to audit from notebook state alone. Now we can use W&B as the source of truth.

For the selected run, we recover:

- the run ID
- the W&B run URL
- the model name
- the dataset version
- the feature count
- shifted-data metrics
- logged artifacts such as confusion matrices, dataset fingerprints, feature schemas, and model files

This is the key workflow: once model tracking is in place, a surprising result becomes something you can investigate instead of something you have to reconstruct from memory.

In [ ]:
selected_run_record = results_df[results_df["run_id"] == best_run_id].iloc[0]
api = wandb.Api()
selected_wandb_run = api.run(selected_run_record["wandb_run_path"])

print("Recovered W&B run")
print("  run_id:", selected_wandb_run.id)
print("  run_url:", selected_wandb_run.url)
print("  model_name:", selected_wandb_run.config.get("model_name"))
print("  dataset_version:", selected_wandb_run.config.get("dataset_version"))
print("  feature_count:", selected_wandb_run.config.get("feature_count"))
print("  bloomberg f1_fake:", selected_wandb_run.summary.get("bloomberg_shift_eval_f1_fake"))
print("  bloomberg recall_fake:", selected_wandb_run.summary.get("bloomberg_shift_eval_recall_fake"))

print("\nLogged artifacts:")
for artifact in selected_wandb_run.logged_artifacts():
    print(" ", artifact.name, "| type:", artifact.type)


In [ ]:
# Download the selected model artifact from W&B and verify it predicts on the shifted evaluation matrix.
model_artifact_ref = selected_run_record["model_artifact_ref"]
print("Selected model artifact:", model_artifact_ref)

model_artifact = api.artifact(model_artifact_ref)
artifact_dir = Path(model_artifact.download(root=str(PROJECT_ROOT / "downloaded_wandb_models" / best_run_id)))
loaded_model = joblib.load(artifact_dir / "model.joblib")

X_shift_eval_for_selected = eval_matrices_v2["bloomberg_shift_eval"][0]
y_shift_eval_for_selected = eval_matrices_v2["bloomberg_shift_eval"][1]

loaded_preds = loaded_model.predict(X_shift_eval_for_selected)
print("Loaded model prediction sample:", loaded_preds[:10])
print("Loaded model accuracy on shifted eval:", accuracy_score(y_shift_eval_for_selected, loaded_preds))


## Part 7: A Small FastAPI Serving Demo

### What does "serving" mean?

Training a model produces an artifact. Serving makes that artifact available for predictions through an interface that other code can call.

W&B is not a model-serving framework. Its role here is to track and version the model artifact that we would deploy somewhere else. The actual serving layer could be FastAPI, BentoML, a cloud endpoint, a batch scoring job, or another production system.

For this notebook, we use a minimal FastAPI app to show the serving boundary:

1. Load the selected model from its W&B artifact reference.
2. Define one prediction function with a small JSON contract.
3. Expose that function through `POST /predict`.
4. Call the endpoint with `curl` from the same notebook runtime.

This is not a public production API. It is a local runtime demo that makes the deployment idea concrete: downstream code should call a stable prediction interface backed by a specific tracked model version.


In [ ]:
model_uri = model_artifact_ref
print("Selected tracked model artifact:")
print(model_uri)

served_model = loaded_model
print("Model loaded from W&B artifact and ready for prediction.")


In [ ]:
from fastapi import FastAPI
import socket
import threading
import uvicorn


def predict_like_serving_endpoint(payload):
    """Prediction contract used by the local FastAPI demo.

    Expected payload shape:
    {
        "dataframe_split": {
            "columns": [...],
            "data": [[...], ...],
            "index": [...]
        }
    }
    """
    if "dataframe_split" not in payload:
        raise ValueError("Payload must contain a 'dataframe_split' key.")

    frame = pd.DataFrame(**payload["dataframe_split"])
    predictions = served_model.predict(frame)
    labels = [INVERSE_LABEL_MAP.get(int(pred), str(pred)) for pred in predictions]
    return {
        "predictions": predictions.tolist(),
        "predicted_labels": labels,
        "model_uri": model_uri,
    }


app = FastAPI(title="Week 5 Misinformation Detection Demo")


@app.get("/")
def root():
    return {
        "status": "ok",
        "message": "Local week 5 serving demo is running.",
        "model_uri": model_uri,
    }


@app.post("/predict")
def predict(payload: dict):
    return predict_like_serving_endpoint(payload)


def port_is_open(host="127.0.0.1", port=8000):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, port)) == 0


if port_is_open():
    print("Port 8000 is already in use. Reusing the existing local server for the curl demo.")
else:
    server_thread = threading.Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
        daemon=True,
    )
    server_thread.start()
    time.sleep(2)
    print("Started local FastAPI server at http://127.0.0.1:8000")

example_payload = {
    "dataframe_split": X_shift_eval_for_selected.head(1).to_dict(orient="split")
}
payload_path = Path("/tmp/week5_payload.json")
payload_path.write_text(json.dumps(example_payload), encoding="utf-8")

print("Saved example request payload to:", payload_path)
print("Direct Python function response:")
predict_like_serving_endpoint(example_payload)


In [ ]:
%%bash
set -e

echo "Health check:"
curl -s http://127.0.0.1:8000/ | python -m json.tool

echo ""
echo "Prediction request:"
curl -s -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  --data-binary @/tmp/week5_payload.json | python -m json.tool


## Key Takeaways and Next Steps

### What We Learned

**1. We reused a known failure case as a tracking problem**
- The Bloomberg/GPT-4o failure was already discovered in week 3 session 4.
- Today, that failure became the reason to introduce experiment tracking.

**2. Tracking is part of production ML**
- A model result depends on data, features, code, parameters, and artifacts.
- If those pieces are not tracked, old results become hard to explain.

**3. Feature schemas need to be explicit**
- The preprocessing steps are familiar, but they still matter for reproducibility.
- Temporary notebook columns can accidentally change training.
- A stable feature list makes retraining and serving safer.

**4. W&B makes tradeoffs visible**
- A model can improve shifted-data fake F1 while losing accuracy on the original FakeNewsNet holdout.
- A single sorted metric is not the same thing as a complete production decision.
- Logged metrics and parameters make these tradeoffs easier to discuss.

**5. W&B connects runs to evidence**
- Metrics show what happened.
- Parameters show how the model was configured.
- Dataset fingerprints show what data was used.
- Artifacts show supporting evidence such as confusion matrices and feature schemas.
- Model artifacts make the selected run reloadable.

**6. Serving needs a stable contract**
- The FastAPI endpoint is intentionally small, but it creates a real HTTP boundary.
- The endpoint uses a specific model artifact recovered from W&B.
- The same tracking pattern helps connect a deployed prediction back to a training run.

**7. The pattern transfers to other projects**
- Stock-price prediction can track market data versions and feature windows.
- Legal RAG can track document snapshots, retriever settings, and evaluation results.
- Any production ML project benefits from knowing which version produced which result.

In the next project step, you can adapt this pattern to your own group project by replacing the data loader, feature builder, model candidates, and evaluation metrics while keeping the tracking structure.
